In [7]:
import os, cv2, numpy as np, matplotlib.pyplot as plt

# Path ke folder hasil ekstrak
input_folder = "/content/images_training.zip"  # masuk ke subfolder 'images'
output_folder = "/content/folder_gambar_enhanced"
os.makedirs(output_folder, exist_ok=True)

# Cari semua gambar di subfolder secara rekursif
valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tif')
image_files = []
for root, dirs, files in os.walk(input_folder):
    for f in files:
        if f.lower().endswith(valid_ext):
            image_files.append(os.path.join(root, f))

if not image_files:
    raise ValueError("Tidak ada file gambar ditemukan. Pastikan file berada di subfolder 'images'.")

# Proses enhancement lanjutan
for img_path in image_files:
    filename = os.path.basename(img_path)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Gagal membaca {filename}, dilewati.")
        continue

    # CLAHE untuk kontras
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # Sharpening
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    enhanced_img = cv2.filter2D(enhanced_img, -1, kernel)

    # Denoising ringan
    enhanced_img = cv2.fastNlMeansDenoisingColored(enhanced_img, None, 5,5,7,21)

    # Simpan hasil
    cv2.imwrite(os.path.join(output_folder, filename), enhanced_img)

print(f"Proses enhancement selesai. Hasil disimpan di: {output_folder}")

# Tampilkan contoh hasil
num_examples = min(5, len(image_files))
plt.figure(figsize=(12, 5))

for i, img_path in enumerate(image_files[:num_examples]):
    orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    enhanced = cv2.cvtColor(cv2.imread(os.path.join(output_folder, os.path.basename(img_path))), cv2.COLOR_BGR2RGB)

    plt.subplot(2, num_examples, i+1)
    plt.imshow(orig)
    plt.title(f"Asli {i+1}")
    plt.axis('off')

    plt.subplot(2, num_examples, i+1+num_examples)
    plt.imshow(enhanced)
    plt.title(f"Enhanced {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

ValueError: Tidak ada file gambar ditemukan. Pastikan file berada di subfolder 'images'.